<a href="https://colab.research.google.com/github/phinnphace/asl-sovereign/blob/main/gemma2training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes

In [ ]:
from unsloth import FastLanguageModel
import torch

# Standard constraints for Colab
max_seq_length = 2048
dtype = None
load_in_4bit = True

# Load Gemma 2 9B Instruct
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/gemma-2-9b-it-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# Apply the LoRA Adapters (the actual part we will be training)
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # The "size" of our fine-tuning. 16 is a solid default.
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

In [ ]:
from unsloth import FastLanguageModel
import torch

# Standard constraints for Colab
max_seq_length = 2048
dtype = None
load_in_4bit = True

# Load Gemma 2 9B Instruct
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/gemma-2-9b-it-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# Apply the LoRA Adapters (the actual part we will be training)
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # The "size" of our fine-tuning. 16 is a solid default.
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

In [ ]:
from datasets import load_dataset

# Load the file we just created
dataset = load_dataset("json", data_files={"train": "/content/asl_training_data.jsonl"}, split="train")

# Define the exact prompt structure Gemma 2 expects
prompt_template = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

# Format the dataset using a mapping function
def format_prompts(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input_text, output in zip(instructions, inputs, outputs):
        # We add the EOS (End of Sequence) token so the model knows when to stop talking!
        text = prompt_template.format(instruction, input_text, output) + tokenizer.eos_token
        texts.append(text)
    return { "text" : texts, }

dataset = dataset.map(format_prompts, batched = True)
print("Dataset loaded and formatted for Gemma!")

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60, # A short run to quickly bake these specific patterns in
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

print("Igniting the training loop...")
trainer_stats = trainer.train()
print("Training Complete! The model now inherently understands your matrix topologies.")

In [ ]:
# Phase 7: Save the Fine-Tuned Adapters
model.save_pretrained("asl_provenance_lora")
tokenizer.save_pretrained("asl_provenance_lora")

print("Adapters saved successfully!")